<a href="https://colab.research.google.com/github/giszang09-art/kim_RAG_memory/blob/main/connect-ai/train-sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `sellstart/kim_GraphRAG_longmomery` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-2-2b-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `sellstart/kim_graphRAG_longmemory_bank` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 52 · seq 1024 · linear · 양자화 q4_k_m (데이터 26개)


In [1]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [2]:
from huggingface_hub import notebook_login
notebook_login()


In [3]:
# 🔐 로그인을 맨 앞에서 확인 — 안 돼 있으면 긴 학습 전에 바로 멈춰서 시간 낭비 방지
from huggingface_hub import HfApi
try:
    print("✅ 로그인됨:", HfApi().whoami()["name"], "— 결과는 내 계정에 올라가요")
except Exception:
    raise SystemExit("❌ 먼저 위 🔑 칸에 HuggingFace write 토큰을 붙여넣고 Login을 누르세요. 그다음 [런타임 → 모두 실행]을 다시 누르면 됩니다.")


✅ 로그인됨: sellstart — 결과는 내 계정에 올라가요


In [4]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

✅ 베이스 모델 로딩 완료


In [5]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
Connect AI 앱이 업로드한 데이터셋. 각 행 = `{conversations:[{user},{assistant}]}`


In [14]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
ds = load_dataset("sellstart/kim_GraphRAG_longmomery", data_files="connect-ai-brain.jsonl", split="train", token=True)
tokenizer = get_chat_template(tokenizer, chat_template="gemma")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])

Unsloth: Will map <end_of_turn> to EOS = <eos>.


Map:   0%|          | 0/26 [00:00<?, ? examples/s]

데이터 개수: 26
<start_of_turn>user
미국 학사·석사 학위를 취득해도 현실적인 인턴십 기회는 어떻게 얻을 수 있나요?<end_of_turn>
<start_of_turn>model
미국 학사·석사 학위를 10년·큰 비용을 들여 땄지만, 지금은 그 졸업장만으로 취업이 어렵다. 인턴 요건조차 "분야 상위 1%·수학 올림피아드 입상" 수준으로 비현실적으로 높아졌다.<end_of_turn>



In [15]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 52, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/26 [00:00<?, ? examples/s]

In [16]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 계열마다 다름 → 실제 텍스트에서 자동 감지 (gemma·llama·qwen 모두 지원)
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
if "<start_of_turn>user" in _t: _im, _rm = "<start_of_turn>user\n", "<start_of_turn>model\n"
elif "<|start_header_id|>" in _t: _im, _rm = "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
elif "<|im_start|>" in _t: _im, _rm = "<|im_start|>user\n", "<|im_start|>assistant\n"
elif "<|turn>user" in _t: _im, _rm = "<|turn>user\n", "<|turn>model\n"
else: _im, _rm = None, None
if _rm:
    trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
    print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 응답만 학습")
else:
    print("ℹ️ 마커 자동감지 실패 → 전체 텍스트로 학습(문제 없음)")


Map (num_proc=5):   0%|          | 0/26 [00:00<?, ? examples/s]

Filter (num_proc=5):   0%|          | 0/26 [00:00<?, ? examples/s]

✅ 마스킹 마커 자동감지: <start_of_turn>model — 응답만 학습


In [17]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26 | Num Epochs = 8 | Total steps = 52
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
1,3.396554
2,3.932819
3,3.407741
4,2.931474
5,3.025160
6,2.730736
7,2.582278
8,2.265894
9,2.086903
10,1.809748


🎉 학습 완료! 최종 loss: 0.8181
💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [18]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    try:
        msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    except Exception:
        msg = [{"role":"user","content":prompt}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    inp = inp.to(model.device)
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


Both `max_new_tokens` (=220) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ 내 사업/지식에 대해 아는 걸 알려줘
💬 ③ —> 내 사업/지식에 대해 아는 게 없다. 같은 사람을 만나며 사업을 할 줄 알아야 한다.
──────────────────────────────────────────────────────────
❓ 너는 무엇을 도와줄 수 있어?
💬 ②Iframe — type text — text = "너는 무엇을 도와줄 수 있어?"
──────────────────────────────────────────────────────────


## 💾 저장 → HuggingFace
**safetensors(AI 진화·합성용) + GGUF(앱 실행용)** 둘 다 올라가요. (맨 앞에서 로그인했으니 바로 됩니다)


In [19]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
# 📤 저장 위치 = "내" HF 계정 (위에서 로그인한 본인 계정으로 자동 — 노트북이 공유돼도 안전)
from huggingface_hub import HfApi
NAME = "kim_graphRAG_longmemory_bank"
OUTPUT = f'{HfApi().whoami()["name"]}/{NAME}'
print("📤 내 계정에 저장:", OUTPUT)
# ① 합성용 safetensors (AI 진화소에서 다시 합칠 수 있어요 — 이게 없으면 합성 불가!)
try:
    model.push_to_hub_merged(OUTPUT, tokenizer, save_method="merged_16bit", token=True)
    print("✅ safetensors 업로드 — AI 진화소에서 합치기 가능")
except Exception as e:
    print("⚠️ 병합 업로드 실패 → 어댑터(LoRA)로 폴백:", e)
    model.push_to_hub(OUTPUT, token=True); tokenizer.push_to_hub(OUTPUT, token=True)
# ② 앱 실행용 GGUF
model.push_to_hub_gguf(OUTPUT, tokenizer, quantization_method="q4_k_m", token=True)
print(f"✅ 완료! safetensors(합성용)+GGUF(실행용) 둘 다 → Connect AI 앱 🤖 내 AI 에서 {OUTPUT} 받기")


📤 내 계정에 저장: sellstart/kim_graphRAG_longmemory_bank


config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in sellstart/kim_graphRAG_longmemory_bank/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gemma-2-2b-it.Q4_K_M.gguf:   1%|          | 14.5MB / 1.71GB            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/sellstart/kim_graphRAG_longmemory_bank
Unsloth: Cleaning up temporary files...
✅ 완료! safetensors(합성용)+GGUF(실행용) 둘 다 → Connect AI 앱 🤖 내 AI 에서 sellstart/kim_graphRAG_longmemory_bank 받기
